In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objs as go

In [ ]:
models=['GNO', 'UNO', 'PINK', 'BROWN', 'VIOLET', # noises (R)
        'AR1_GNO', 'STAR_GNO', #    ARMA (R)
        'ARNOLD', 'CHIRIKOV', 
        'OU', 'Oscillator',# Conservative chaotic maps (R)
        'BRW_cont', 
        'HENR_diverse', 'HENR_same', 'QUADRATIC_RSUM', # Sum of frwd and bckwd realisations of chaotic maps (R)
        'AR1_UNO', 'ARMA11_UNO', 'AR3_Gamma', 'N_AR2', 'SETAR1_GNO', 'SETAR2_GNO',
        'HEN', 'HEN_SUM', 'LOGISTIC4', 'LOGISTIC38284', 'QUADRATIC', # Chaotic maps (I)
        'MODA', 'LLOG', # Other deterministic (I)
        'SINE_STOCH', # Other stochastic (I)
        'LORENZ_SUM', 'ROSSLER_SUM', 
        'MACKEYGLASS17', 'VDP', 
        'LORENZ_STOCH_SUM',
        'VDP_STOCH'
        ]

print('Models:', models)

model_keywords={ # discrete
                'GNO': 'reversible', 'UNO': 'reversible', 'PINK': 'reversible', 'BROWN': 'reversible', 'VIOLET': 'reversible',
                'AR1_GNO': 'reversible', 'STAR_GNO': 'reversible',
                'ARNOLD': 'reversible', 'CHIRIKOV': 'reversible',
                'HENR_diverse': 'reversible', 'HENR_same': 'reversible', 'QUADRATIC_RSUM': 'reversible',
                'AR1_UNO': 'irreversible', 'ARMA11_UNO': 'irreversible', 'AR3_Gamma': 'irreversible', 'N_AR2': 'irreversible', 'SETAR1_GNO': 'irreversible', 'SETAR2_GNO': 'irreversible', 
                'HEN': 'irreversible', 'HEN_SUM': 'irreversible', 'LOGISTIC4': 'irreversible', 'LOGISTIC38284': 'irreversible','QUADRATIC': 'irreversible',
                'MODA': 'irreversible', 'LLOG': 'irreversible',
                'SINE_STOCH': 'irreversible',
                # continuous
                'BRW_cont': 'reversible', 'OU': 'reversible', 'Oscillator': 'reversible',
                'LORENZ_SUM': 'irreversible', 'ROSSLER_SUM': 'irreversible',
                'MACKEYGLASS17': 'irreversible', 'VDP': 'irreversible',
                'LORENZ_STOCH_SUM': 'irreversible',
                'VDP_STOCH': 'irreversible'
        }


In [ ]:
def plot_violin_models(df, models, op, model_labels):
    
    colors={'reversible': 'blue', 'irreversible': 'red'}

    data=[]
    # Extract rows whose index is a multi iindex but that contains model
    for model in models:
        df_model = df.loc[model]
        array_op= np.array(df_model[op])
        data.append(array_op)


    fig = go.Figure()

    for i, model in enumerate(models):
        fig.add_trace(go.Violin(y=data[i], name=model, marker=dict(color=colors[model_labels[model]]), 
                                box_visible=True, meanline_visible=True, points='all', 
                                jitter=0.25, pointpos=-1.5, marker_size=2, line_width=1.2))

    fig.update_layout(title=op, width=1200, height=600,
                    plot_bgcolor='whitesmoke',  
                    paper_bgcolor='white', 
                    xaxis=dict(
                    showgrid=True,
                    tickfont=dict(size=14),
                    gridwidth=1),
                    yaxis=dict( 
                    showgrid=True,
                    tickfont=dict(size=18),
                    gridwidth=1),
                    title_font=dict(size=26),
                    showlegend=False)
    
    fig.show()

In [ ]:
# folder where this script lives
BASE_DIR = Path.cwd()
path_base = str(BASE_DIR)

REPO_DIR = BASE_DIR.parents[2]

DATA_DIR = REPO_DIR / 'data-tr'
print(DATA_DIR)
path_data= str(DATA_DIR)


# Directories to save data 
dir_zero= path_base + '/data-zero/'

# Directory for supplementary materials
dir_supp= path_base + '/supplementary-material/'

# Directory to find hctsa pre-processed hctsa data
dir_hctsa= path_data + '/main-analysis/data-analysis/'

# Directory to save accuracy results
dir_accuracy= path_base + '/data-analysis/accuracy/'

os.makedirs(dir_accuracy, exist_ok=True)

# Directory to save figures
dir_figures = path_base + '/data-analysis/figures/'

os.makedirs(dir_figures, exist_ok=True)

In [ ]:
# Load good performing features
df_ops_good = pd.read_csv(dir_accuracy+'df_accuracy_good_1NN.csv')
# Load ops zero
df_ops_zero = pd.read_csv(dir_zero+'df_ops_zero.csv')

# Check what good features are among the zero ones
common_values = pd.merge(df_ops_good, df_ops_zero, left_on='Operation', right_on='Name', how='inner')
# Display the common values
print(common_values)

In [ ]:
# Load only features that are not zero
df_ops_good = df_ops_good[~df_ops_good['Operation'].isin(df_ops_zero['Name'])]
df_good_hctsa = pd.read_csv(dir_accuracy+'df_good_hctsa_1NN.csv')
df_good_hctsa.set_index('Model', inplace=True)

# Select rows with model in model
df_good_hctsa = df_good_hctsa[df_good_hctsa.index.get_level_values(0).isin(models)]

In [ ]:
# LOAD IF I WANT THE ACCURACY ON THE FULL SET OF OPERATIONS AND DATASET
# load accuracy 1NN
df_ops_full = pd.read_csv(dir_accuracy+'df_accuracy_abs_1NN.csv')

# Load full hctsa matrix
df_full_hctsa=pd.read_csv(dir_hctsa+'df_TS_DataMat_diff.csv')
df_full_hctsa.set_index(['Model'], inplace=True)

# Add column with reversible/irreversible label
df_type=df_full_hctsa.index.get_level_values(0).map(model_keywords)
df_full_hctsa.insert(0,'Type',df_type)


In [ ]:
# Load only features that are not zero
df_ops_good = df_ops_good[~df_ops_good['Operation'].isin(df_ops_zero['Name'])]

In [ ]:
op='MF_steps_ahead_ar_best_6_mabserr_1'
# MF_armax_2_2_05_1_normksstat
plot_violin_models(df_good_hctsa, models, op, model_keywords)
['SB_TransitionMatrix_51_symsumdiff', 'SB_TransitionMatrix_41_symsumdiff']

In [ ]:
# Scatter plot of the values of hctsa for the first two features in df_ops_good
# Select columns with name of features 'AC_nl_001' and 'AC_nl_002'
feat1 = 'SB_MotifTwo_diff_u'
feat2 = 'AC_nl_001'

# Prepare the two-feature dataframe
df_two = df_ops_good[df_ops_good['Operation'].isin([feat1, feat2])]
df_two_hctsa = df_good_hctsa[df_good_hctsa.columns[df_good_hctsa.columns.isin(df_two['Operation'])]]

# Add model types
df_two_hctsa['Type'] = df_two_hctsa.index.get_level_values(0).map(model_keywords)

# Colors for types
color_map = {'reversible': 'blue', 'irreversible': 'red'}
colors = df_two_hctsa['Type'].map(color_map)

# --- Matplotlib scatter plot ---
plt.figure(figsize=(12, 6))
plt.scatter(df_two_hctsa[feat1], df_two_hctsa[feat2],
            c=colors, s=40, alpha=0.8)

# Labels, title
plt.xlabel(feat1, fontsize=14)
plt.ylabel(feat2, fontsize=14)
plt.title(f"{feat1} vs {feat2}", fontsize=24)

# Background and grid
plt.grid(True, linewidth=0.8)
plt.gca().set_facecolor("whitesmoke")

# Legend
for t, c in color_map.items():
    plt.scatter([], [], c=c, label=t)
plt.legend(title="Type of model", fontsize=12, title_fontsize=13)

# Add annotations every 100 points
for i in range(0, len(df_two_hctsa), 100):
    x = df_two_hctsa.iloc[i][feat1]
    y = df_two_hctsa.iloc[i][feat2]
    label = df_two_hctsa.index[i]
    plt.annotate(label, (x, y),
                 textcoords="offset points", xytext=(0, -25),
                 ha='center', fontsize=9, fontfamily='Arial',
                 arrowprops=dict(arrowstyle="->", lw=0.8))

plt.tight_layout()
plt.show()


In [ ]:
# Write to Excel with formatting
os.makedirs(dir_supp, exist_ok=True)
with pd.ExcelWriter(dir_supp+'Supplement_Table_S3.xlsx', engine='xlsxwriter') as writer:
    df_ops_good.to_excel(writer, index=False, sheet_name='Sheet1')

    workbook  = writer.book
    worksheet = writer.sheets['Sheet1']

    # Define formats
    format_grey = workbook.add_format({'bg_color': '#DDDDDD'})
    format_white = workbook.add_format({'bg_color': '#FFFFFF'})

    # Specify the header names in bold
    bold_format = workbook.add_format({'bold': True, 'font_color': '#000000', 'bg_color': '#DDDDDD'})
    header_names = ['Name', 'Global average', 'Feature class', 'Accuracy']
    # Write the header with bold format
    for col_num, header in enumerate(df_ops_good.columns):
        worksheet.write(0, col_num, header, bold_format)


    # Apply alternating colors row by row
    for row in range(1, len(df_ops_good)+1):  # Start at 1 to skip header
        row_format = format_grey if row % 2 == 0 else format_white
        worksheet.set_row(row, 20, row_format)
    worksheet.set_column('A:A', 40)  # Set width for the first column
    worksheet.set_column('B:B', 20)  # Set width for the second column
    worksheet.set_column('C:C', 20)  # Set width for the third column
